
Gizli katmanı ve çıkış katmanını kur: embedding'leri düzleştir, W1 ve b1 ile tanh, W2 ve b2 ile logits. Loss'u geçen haftaki gibi elle hesapla, sonra F.cross_entropy ile aynı sonucu aldığını göster ve neden onu tercih ettiğimizi videodan anla.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

block_size = 3
def build_dataset(words):  
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  return torch.tensor(X), torch.tensor(Y)
  
X,Y = build_dataset(words)

C = torch.randn((27, 2))
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

### Embedding'leri Düzleştirme: `torch.cat`, `unbind` ve `view`

In [2]:
print(torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape)
print(torch.cat(torch.unbind(emb, 1), 1).shape)
print(emb.view(-1, 6).shape)
#storage için yeni bir değişken oluşturmaz
torch.equal(torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1), emb.view(-1, 6))

torch.Size([228146, 6])
torch.Size([228146, 6])
torch.Size([228146, 6])


True

### Hidden Layer

In [3]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [4]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h.shape

torch.Size([228146, 100])

### Output Layer

In [5]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [6]:
logits = h @ W2 + b2
logits.shape

torch.Size([228146, 27])

### Softmax + NLL

In [7]:
# 32 örneklik dilim üzerinde elle softmax ve negative log likelihood:
counts = logits[:32].exp()
prob = counts / counts.sum(1, keepdims=True)
loss_manual = -prob[torch.arange(32), Y[:32]].log().mean()
loss_manual.item()

19.056737899780273

### F.cross_entropy
gereksiz nesne üretimini engelleyerek depolamma sorunundan kurtarır ve 
exp() tasma sorununu önler

In [8]:
loss_builtin = F.cross_entropy(logits[:32], Y[:32])
print("manual: ", loss_manual.item())
print("cross_entropy:", loss_builtin.item())

manual:  19.056737899780273
cross_entropy: 19.056739807128906
